# Data Health Audit

## Important Note

This notebook `/notebooks/3_data_health_audit_global_only.ipynb` is different than the sub-sampled version `/notebooks/3_data_health_audit_sample_only.ipynb` in terms of structure and purpose. 

Purpose: The sub-sampled version aims to check the data health and explore potential issue in various aspect of the dataset. This one aims to verify if the issues/trend/patterns found in the sub-sampled counterpart will be similar to this version. The similar pattern exhibits from two notebooks will indirectly indicates that the subsampling strategy is unbiased and effective, and vice versa.

Structure: The subsampled version will explicitly write down the logic as you audit the data health, whereas this version will put most workflows in `/src/utils/data_health_audit_helper.py`, and the audit logic will call a specific function to perform and verify health check. 

## Import Modules

In [1]:
from pathlib import Path
import sys
import os

In [2]:
## Use for import external directory modules
# Make sure you are the parent directory
# Define directory path: notebooks/ -> project root
PROJECT_ROOT = Path.cwd().resolve().parent
display(PROJECT_ROOT)

# Add it to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PosixPath('/expanse/lustre/projects/uci157/bguo3/DSC_288R')

In [3]:
# Other libs
from src.utils.pyspark_helper import (
    create_spark_session, 
    memory_count
)
from src.utils.data_health_audit_helper import (
    structure_report,
    schema_audit,
    null_report,
    consistency_report,
    validity_report,
    anomaly_report,
    noise_report,
    uniqueness_report,
    duplicate_report,
    print_section,
    format_report
)

Matplotlib created a temporary cache directory at /scratch/bguo3/job_48715834/matplotlib-x68xymia because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Set up for Spark

In [4]:
# Set up a dedicated job-running spill storage directory
SPARK_LOCAL_DIR = f"/scratch/{os.environ['USER']}/job_{os.environ['SLURM_JOB_ID']}/spark-local"
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

# Set up for Spark app & resource allocation
spark = create_spark_session(
    "steam_reviews_data_health_audit", 
    extra_configs={
        "spark.local.dir": SPARK_LOCAL_DIR
    }
)
print(spark.conf.get("spark.local.dir"))

/scratch/bguo3/job_48715834/spark-local


## Define Paths & Read Full Data Parquet File

In [5]:
# -----------------------------
# Paths
# -----------------------------
EXPANSE_ROOT = "/expanse/lustre/scratch/bguo3/temp_project"
FULL_PARQUET_PATH = f"file:{EXPANSE_ROOT}/steam_reviews/full_parquet"

df = spark.read.parquet(FULL_PARQUET_PATH)
print(f"Read full parquet successfully from: {FULL_PARQUET_PATH}")

Read full parquet successfully from: file:/expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet


## Dataset Structure

In [6]:
ROW_COUNT = df.count()

## Shape
# 1. (# of rows, # of columns, # of cells)
print_section("1. (# of rows, # of columns, # of cells)")
display(structure_report(df, ROW_COUNT))

## 2. Concise summary on columns details
print_section("2. Concise summary on columns details")
display(schema_audit(df))
df.printSchema()

# 3. Memory count
print_section("3. Memory count")
memory_count(df)


1. (# of rows, # of columns, # of cells)


,row_count,column_count,cell_count
0,113885601,18,2049940818



2. Concise summary on columns details


,column_name,present_in_df,expected_type,actual_type,matches_expected_family
0,author_steamid,True,bigint,bigint,True
1,appid,True,int,int,True
2,author_num_games_owned,True,int,int,True
3,author_num_reviews,True,int,int,True
4,author_playtime_forever,True,int,int,True
5,author_playtime_last_two_weeks,True,int,int,True
6,author_playtime_at_review,True,int,int,True
7,author_last_played,True,bigint,bigint,True
8,review,True,string,string,True
9,voted_up,True,boolean,boolean,True


root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- language: string (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)


3. Memory count
Total estimated size: 55.74 GB


## Completeness

In [9]:
## Compeleteness helps us investigate missing value and if we have useful
## features for prediction problems

# 1. Identify missing values per column
# 2. Calculate percentage of missing values
null_report(df, ROW_COUNT).show(truncate=False)

+------------------------------+----------+---------+---------------------+
|column_name                   |null_count|row_count|null_rate            |
+------------------------------+----------+---------+---------------------+
|voted_up                      |2956554   |113885601|0.02596073580891056  |
|timestamp_created             |2955600   |113885601|0.02595235898171183  |
|timestamp_updated             |2414900   |113885601|0.021204612161637538 |
|votes_up                      |1772064   |113885601|0.015560035548304303 |
|votes_funny                   |1554643   |113885601|0.013650917994453048 |
|written_during_early_access   |1423070   |113885601|0.012495609519591507 |
|weighted_vote_score           |1378404   |113885601|0.012103408928754743 |
|comment_count                 |1233750   |113885601|0.010833239576968119 |
|author_steamid                |1878      |113885601|1.649023215849737E-5 |
|author_num_games_owned        |1860      |113885601|1.6332178815125188E-5|
|author_num_

## Consistency & Validity

In [10]:
## Consistency helps us question "Do columns agree with each other logically"?

# 1. timestamp_created should usually be <= timestamp_updated
# 2. playtime_at_review & playtime_last two weeks should be less than lifetime playtime
# 3. all user in the dataset should have at least 1 numbers of review
# 4. author_playtime_forever at later review time should not be lower than earlier review time
format_report(consistency_report(df, ROW_COUNT))


timestamp_created_gt_timestamp_updated
Violation count: 67
Violation percentage: 5.88309666996445e-07
Showing first 10 rows:
+-----------------+-------+-----------------+-----------------+
|author_steamid   |appid  |timestamp_created|timestamp_updated|
+-----------------+-------+-----------------+-----------------+
|76561198059619745|281990 |4                |3                |
|1590801716       |NULL   |1                |0                |
|76561199124521359|686810 |2                |1                |
|76561198010909133|906100 |2                |1                |
|NULL             |NULL   |1                |0                |
|76561198050620715|931690 |1559566943       |1559566933       |
|76561198004160508|1063730|2                |1                |
|76561198021406864|1250410|200              |100              |
|76561198116916435|1255630|18               |17               |
|76561198367452421|314160 |40               |30               |
+-----------------+-------+---------------

In [8]:
## Validity helps us question "Do columns within reasonable range"?

# 5. Count-like columns should not be negative
# 6. weighted_vote_score should be between 0 and 1
format_report(validity_report(df, ROW_COUNT))


author_num_games_owned_negative
Violation count: 0
Violation percentage: 0.0
Showing first 10 rows:
+--------------+-----+----------------------+-----------------+
|author_steamid|appid|author_num_games_owned|timestamp_created|
+--------------+-----+----------------------+-----------------+
+--------------+-----+----------------------+-----------------+


author_num_reviews_negative
Violation count: 0
Violation percentage: 0.0
Showing first 10 rows:
+--------------+-----+------------------+-----------------+
|author_steamid|appid|author_num_reviews|timestamp_created|
+--------------+-----+------------------+-----------------+
+--------------+-----+------------------+-----------------+


author_playtime_forever_negative
Violation count: 0
Violation percentage: 0.0
Showing first 10 rows:
+--------------+-----+-----------------------+-----------------+
|author_steamid|appid|author_playtime_forever|timestamp_created|
+--------------+-----+-----------------------+-----------------+
+------

## Anomaly & Outliers

In [12]:
## Anomaly indicates suspicious value, not automatically wrong
ano_report = anomaly_report(df, ROW_COUNT)

# 1. Check for outlier value using summary statistics
display(ano_report.get('numeric_describe_df').toPandas())

# 2. Create report regarding anomaly report for suspicious features (1) votes_funny artifact
# 3. Check for instances where votes_funny reached max value
# 4. Check for instances where votes_funny near max value
format_report(ano_report)

# 5. Find the largest vote_funny value that follows right after ARTIFACT_THRESHOLD, compute its propoertion comparing to max value
# If it's between 60% - 95%, it will indicate broader range of artifact from API and we will need to dive deeper
print_section("Largest non-artifact votes_funny value below threshold")
ano_value, ano_percentage = ano_report.get('vote_max_non_artifact', (None, None))
if ano_value is not None: print(ano_value)
if ano_percentage is not None: print(ano_percentage)
del ano_report, ano_value, ano_percentage

,summary,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,votes_up,votes_funny,comment_count,weighted_vote_score
0,count,113883741,113883849,113884452,113884453,113884463,112113537,112330958,112651851,112507197
1,mean,833.5436838169902,399.9623533447662,15573.124560111155,116.18008222772954,7129.382843241751,8997958.327789998,6953203.962033075,4467856.069244153,NaN
2,stddev,981228.2934024984,715900.9208064852,496019.5376433794,218722.14813122991,227509.99698184888,1.1864653059586446E8,1.052227155457046E8,8.369630425056835E7,NaN
3,min,0,0,0,0,0,-2016,-53,-96,-89.0
4,max,1651393883,1651393883,1403342723,1393862250,1402630204,1699017189,4294967295,1699015475,NaN



playtime_2w_near_limit
Violation count: 16632
Violation percentage: 0.00014604128927589362
Showing first 10 rows:
+-----------------+------+------------------------------+-----------------------+-----------------+
|author_steamid   |appid |author_playtime_last_two_weeks|author_playtime_forever|timestamp_created|
+-----------------+------+------------------------------+-----------------------+-----------------+
|NULL             |NULL  |1393862250                    |1393862250             |0                |
|NULL             |NULL  |1356627477                    |NULL                   |0                |
|NULL             |NULL  |1290286146                    |1290286146             |0                |
|76561199529253953|730   |112370                        |105504                 |1692376470       |
|76561199404792985|730   |56748                         |203490                 |1666940962       |
|76561199488329913|730   |45285                         |114381                 |1691

## Noise

In [7]:
## Noises are values that are valid but possibly not useful & unstable

# 1. Check for columns with mostly zeros
noise_report(df, ROW_COUNT).show(truncate=False)

+------------------------------+-------------------+---------+--------------------+
|column_name                   |zero_or_false_count|row_count|zero_or_false_rate  |
+------------------------------+-------------------+---------+--------------------+
|comment_count                 |107639931          |113885601|0.9451583874944823  |
|author_playtime_last_two_weeks|100949251          |113885601|0.8864092573037394  |
|votes_funny                   |97979357           |113885601|0.8603313864059075  |
|votes_up                      |77654621           |113885601|0.6818651376305245  |
|weighted_vote_score           |73910148           |113885601|0.6489858889184771  |
|author_num_games_owned        |56725784           |113885601|0.49809443425600397 |
|author_playtime_at_review     |1871739            |113885601|0.016435255937227746|
|author_playtime_forever       |1736863            |113885601|0.015250944673857408|
|author_num_reviews            |634                |113885601|5.566989983220

## Uniqueness & Duplicates

In [13]:
## Uniqueness of catgorical features tells us insight of categories, while for numerical features,
## it indicates potential few_unique_values_per_column issue or possibility for discretization to a categorical type

# 1. Numbers of unique values for categorical/numerical & ID columns
uniqueness_report(df, ROW_COUNT).show(truncate=False)

## Duplicate tells us how many rows has exact duplicates and how frequent would a user to review the same game

# 2. Numbers of duplicate found in dataframe
# 3. Does the same user appear to have created multiple review records for the same game at the same time?
# 4. Inspect more details ("author_steamid", "appid", "timestamp_created") regarding instances in #3
# 5. Print out the duplicate reviews
dup_reports = duplicate_report(df, ROW_COUNT)
for (report, df) in zip(dup_reports['summary_rows'], dup_reports['issue_dfs'].values()):
    print_section(report['issue_name'])
    for name, val in report.items():
        if name == 'issue_name': continue
        print(f"{name}: {val}")
    df.show(n=5, truncate=False)
         
# 6. Create report on metadata difference on same (author ID, game ID, review date created)
# UN-ACHIEVE-ABLE; reason: too much join, putting too pressure on spark worker memory

+------------------------------+--------+---------------------+
|column_name                   |n_unique|unique_rate          |
+------------------------------+--------+---------------------+
|voted_up                      |2       |1.7561482596908805E-8|
|written_during_early_access   |2       |1.7561482596908805E-8|
|author_num_reviews            |1334    |1.1713508892138172E-5|
|author_num_games_owned        |9810    |8.613907213783769E-5 |
|author_playtime_last_two_weeks|19931   |1.750089548194947E-4 |
|comment_count                 |320438  |0.0028136831801941317|
|author_playtime_at_review     |359479  |0.00315649210122709  |
|votes_funny                   |493936  |0.004337124233993374 |
|author_playtime_forever       |544273  |0.004779120408733673 |
|votes_up                      |643808  |0.005653111493875332 |
|weighted_vote_score           |5979642 |0.05250568945937248  |
+------------------------------+--------+---------------------+


exact_dup
row_count: 113885601
distinc

In [ ]:
spark.stop()